# CNOT

Adapted from Qualtran v0.7.0 commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from IPython.display import SVG, display

from examples.algorithms.cnot_tutorial import cnot_graph, cnot_score_data, cnot_signature, cnot_truth_table
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

## `CNOT`
Two-qubit controlled-NOT.

#### Registers
 - `ctrl`: One-bit control register.
 - `target`: One-bit target register.


Both ports are THRU, ordered control then target. The operation is atomic and self-adjoint. Its truth table, tensor factors and instruction body use their existing native owners.

In [ ]:
from flagquantum.compiler.registers.primitives.controlled_not import cnot_with_controls_ir

### Example Instances

In [ ]:
cnot = cnot_signature()
cnot

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("CNOT", cnot, {})])))

### Call Graph

In [ ]:
graph = cnot_graph()
display(SVG(render_call_graph_svg(export_call_graph_dot(
    graph.root, graph.direct_calls, title=lambda node: "CNOT",
))))
print("Atomic leaves", graph.leaf_counts)

## Truth table, port glyphs and coherent execution

The same CNOT typed body supplies the signature and the truth table. The score marks a filled control circle and a plus target. Append one native CX to prepare a Bell state from a plus and zero. Appending it again restores the original state. There are zero measurements and zero work wires.

In [ ]:
from math import sqrt
import torch
from examples.algorithms.binary_field_arithmetic import FieldOperation
from flagquantum.circuit import Circuit

print(cnot_truth_table())
score = cnot_score_data()
assert [p["symb_cls"] for p in score["soqs"] if p["seq_x"] == 0] == ["Circle", "ModPlus"]
gates = cnot_with_controls_ir(0, 1)
circuit = Circuit(2, dtype=torch.complex128)
circuit.h(0)
initial = circuit.state().flatten().clone()
operation = FieldOperation(2, gates)
circuit.append(operation)
expected = torch.tensor([1, 0, 0, 1], dtype=torch.complex128) / sqrt(2)
torch.testing.assert_close(circuit.state().flatten(), expected)
circuit.append(operation)
torch.testing.assert_close(circuit.state().flatten(), initial)
print("One emitted CX; zero measurements and zero work wires")

## COPY/XOR factorization and resource count

The contraction uses the two rank-three factors for each call in a three-call chain. One logical CNOT is one Clifford in the source cost model. COPY and XOR are tensor data, while the emitted operation is a single CX.

In [ ]:
from examples.algorithms.factorized_controlled_not import main
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from examples.algorithms.cnot_tutorial import CNOT

main()
cost = query_qec_gate_cost(CNOT)
assert (cost.clifford, cost.t, cost.measurement) == (1, 0, 0)
print("Source logical model", cost)